# 04 - Selected citation source share check

This notebook makes a small check figure for selected PC member cases.
My goal is to see whether, during the selected PC service year, a larger
share of the researcher's studied conference citations comes from the same
conference.

## 1. Setup

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

import os
import sys
from pathlib import Path

os.environ.setdefault("ARROW_USER_SIMD_LEVEL", "NONE")

start = Path.cwd().resolve()
project_candidate = None
for candidate in [start, *start.parents]:
    if (candidate / "config" / "project_config.yaml").exists():
        project_candidate = candidate
        break
    child_matches = sorted(
        child for child in candidate.iterdir()
        if child.is_dir() and (child / "config" / "project_config.yaml").exists()
    )
    if len(child_matches) == 1:
        project_candidate = child_matches[0]
        break

if project_candidate is None:
    raise FileNotFoundError("Could not find config/project_config.yaml")

if str(project_candidate) not in sys.path:
    sys.path.insert(0, str(project_candidate))

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib import font_manager

from project_setup import ensure_dirs, setup_project

setup = setup_project(project_candidate)
PROJECT = setup.project_folder

STEP_3_PREPARED = PROJECT / "step_3_data" / "prepared"
STEP_4_SUMMARY = PROJECT / "step_4_artifacts" / "summary_tables"
MAIN_TEXT_FIGURES = PROJECT / "step_4_artifacts" / "main_text_figures"
REPORT_FIGURES = PROJECT / "report_latex" / "figures"
ensure_dirs(STEP_4_SUMMARY, MAIN_TEXT_FIGURES, REPORT_FIGURES)

PANEL_PATH = STEP_3_PREPARED / "panel.parquet"
SELECTED_CASES_PATH = STEP_4_SUMMARY / "selected_spike_and_fade_cases.csv"

SELECTED_SOURCE_SHARE_ROWS_OUT = (
    STEP_4_SUMMARY / "selected_pc_source_share_analysis_rows.csv"
)
SELECTED_SOURCE_SHARE_SUMMARY_OUT = (
    STEP_4_SUMMARY / "selected_source_share_summary.csv"
)
SELECTED_SOURCE_SHARE_FIGURE_OUT = (
    MAIN_TEXT_FIGURES / "selected_pc_source_share_analysis.pdf"
)
REPORT_SELECTED_SOURCE_SHARE_FIGURE_OUT = (
    REPORT_FIGURES / "selected_pc_source_share_analysis.pdf"
)
SELECTED_SOURCE_DISTRIBUTION_ROWS_OUT = (
    STEP_4_SUMMARY / "selected_pc_source_distribution_rows.csv"
)
EVENT_TIMES = [-2, -1, 0, 1, 2]

print("Project folder: .")
print(f"Run mode: {setup.run_mode}")
print(f"Overwrite artifacts: {setup.overwrite_artifacts}")

Project folder: .
Run mode: fast
Overwrite artifacts: True


## 2. Plot style

In [2]:
font_path = PROJECT / "fonts" / "LinLibertine_R.ttf"
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    font_prop = font_manager.FontProperties(fname=font_path)
    font_family = font_prop.get_name()
else:
    font_family = "serif"

mpl.rcParams.update(
    {
        "axes.titlesize": 10,
        "axes.labelsize": 10,
        "font.size": 10,
        "legend.fontsize": 9,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "font.family": font_family,
        "text.usetex": True,
        "axes.linewidth": 0.8,
        "axes.edgecolor": "#333333",
        "xtick.direction": "out",
        "ytick.direction": "out",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

CONFERENCE_COLORS = {
    "POPL": "#81c8be",
    "ICFP": "#f5bde6",
    "OOPSLA": "#8aadf4",
    "OOPSLA_COMBINED": "#8aadf4",
    "PLDI": "#e5c890",
}
SOFT_BLACK = "#333333"
PC_YEAR_RED = "#b23a3a"
ROW_SEPARATOR = "#d9d9d9"
LIGHT_GRID = "#DDDDDD"

## 3. Read selected cases and panel data

In [3]:
panel = pd.read_parquet(PANEL_PATH)
selected_cases = pd.read_csv(SELECTED_CASES_PATH)

panel["year"] = panel["year"].astype(int)
panel["citation_count"] = pd.to_numeric(panel["citation_count"], errors="coerce")
selected_cases["pc_year"] = selected_cases["pc_year"].astype(int)

selected_case_keys = [
    ("clementpitclaudel", "PLDI", 2024),
    ("clementpitclaudel", "OOPSLA_COMBINED", 2025),
    ("viktorkuncak", "OOPSLA", 2020),
    ("viktorkuncak", "PLDI", 2021),
    ("peterthiemann", "ICFP", 2023),
    ("simonjgay", "ICFP", 2023),
    ("ezgicicek", "ICFP", 2019),
    ("brandonlucia", "OOPSLA", 2019),
    ("jonathanbrachthauser", "POPL", 2024),
    ("fritzhenglein", "POPL", 2024),
]

key_frame = pd.DataFrame(
    selected_case_keys,
    columns=["researcher_id", "conference", "pc_year"],
)
selected_subset = key_frame.merge(
    selected_cases,
    on=["researcher_id", "conference", "pc_year"],
    how="left",
    validate="one_to_one",
)

if selected_subset["name"].isna().any():
    display(selected_subset[selected_subset["name"].isna()])
    raise ValueError("Some requested selected cases were not found.")

selected_subset["plot_order"] = range(1, len(selected_subset) + 1)
display(
    selected_subset[
        ["plot_order", "name", "conference", "pc_year", "selected_event_label"]
    ]
)

,plot_order,name,conference,pc_year,selected_event_label
0,1,Clément Pit-Claudel,PLDI,2024,PLDI 2024
1,2,Clément Pit-Claudel,OOPSLA_COMBINED,2025,OOPSLA 2025
2,3,Viktor Kunčak,OOPSLA,2020,OOPSLA 2020
3,4,Viktor Kunčak,PLDI,2021,PLDI 2021
4,5,Peter Thiemann,ICFP,2023,ICFP 2023
5,6,Simon J. Gay,ICFP,2023,ICFP 2023
6,7,Ezgi Çiçek,ICFP,2019,ICFP 2019
7,8,Brandon Lucia,OOPSLA,2019,OOPSLA 2019
8,9,Jonathan Immanuel Brachthäuser,POPL,2024,POPL 2024
9,10,Fritz Henglein,POPL,2024,POPL 2024


## 4. Compute selected conference source shares

For each selected researcher--PC year case, I compute the selected conference
citation share for each event time. The numerator is the citation count from
the selected conference in that calendar year. The denominator is the total
citation count from all studied conferences in that same calendar year.

In [4]:
def selected_source_conferences(conference):
    if conference == "OOPSLA_COMBINED":
        return {"OOPSLA1", "OOPSLA2"}
    return {conference}


def build_selected_source_share_rows(panel_source, selected_source):
    records = []
    for _, selected in selected_source.iterrows():
        source_conferences = selected_source_conferences(selected["conference"])
        for event_time in EVENT_TIMES:
            year = int(selected["pc_year"]) + event_time
            researcher_year = panel_source.loc[
                panel_source["researcher_id"].eq(selected["researcher_id"])
                & panel_source["year"].eq(year)
            ].copy()

            if researcher_year.empty:
                total_citations = np.nan
                selected_citations = np.nan
            else:
                total_citations = researcher_year["citation_count"].sum()
                selected_citations = researcher_year.loc[
                    researcher_year["conference"].isin(source_conferences),
                    "citation_count",
                ].sum()

            source_share = (
                selected_citations / total_citations
                if pd.notna(total_citations) and total_citations > 0
                else np.nan
            )
            records.append(
                {
                    "plot_order": selected["plot_order"],
                    "researcher_id": selected["researcher_id"],
                    "name": selected["name"],
                    "conference": selected["conference"],
                    "plot_conference": selected["plot_conference"],
                    "pc_year": selected["pc_year"],
                    "selected_event_label": selected["selected_event_label"],
                    "event_time": event_time,
                    "year": year,
                    "selected_source_conferences": ", ".join(
                        sorted(source_conferences)
                    ),
                    "selected_conference_citations": selected_citations,
                    "all_studied_conference_citations": total_citations,
                    "source_share": source_share,
                    "source_share_pct": 100 * source_share
                    if pd.notna(source_share)
                    else np.nan,
                }
            )
    rows = pd.DataFrame(records)
    baseline = rows.loc[rows["event_time"].eq(-2), [
        "researcher_id",
        "conference",
        "pc_year",
        "source_share",
        "source_share_pct",
    ]].rename(
        columns={
            "source_share": "baseline_source_share",
            "source_share_pct": "baseline_source_share_pct",
        }
    )
    rows = rows.merge(
        baseline,
        on=["researcher_id", "conference", "pc_year"],
        how="left",
        validate="many_to_one",
    )
    rows["delta_source_share"] = rows["source_share"] - rows["baseline_source_share"]
    rows["delta_source_share_pp"] = 100 * rows["delta_source_share"]
    return rows


selected_source_share_rows = build_selected_source_share_rows(panel, selected_subset)

pc_year_summary = selected_source_share_rows.loc[
    selected_source_share_rows["event_time"].eq(0)
].copy()
pc_year_summary = pc_year_summary[
    [
        "plot_order",
        "name",
        "conference",
        "pc_year",
        "selected_conference_citations",
        "all_studied_conference_citations",
        "source_share_pct",
        "baseline_source_share_pct",
        "delta_source_share_pp",
    ]
].sort_values("plot_order")

display(
    pc_year_summary.assign(
        source_share_pct=lambda d: d["source_share_pct"].round(1),
        baseline_source_share_pct=lambda d: d["baseline_source_share_pct"].round(1),
        delta_source_share_pp=lambda d: d["delta_source_share_pp"].round(1),
    )
)

,plot_order,name,conference,pc_year,selected_conference_citations,all_studied_conference_citations,source_share_pct,baseline_source_share_pct,delta_source_share_pp
2,1,Clément Pit-Claudel,PLDI,2024,13.0,20.0,65.0,35.3,29.7
7,2,Clément Pit-Claudel,OOPSLA_COMBINED,2025,9.0,20.0,45.0,16.7,28.3
12,3,Viktor Kunčak,OOPSLA,2020,20.0,37.0,54.1,16.7,37.4
17,4,Viktor Kunčak,PLDI,2021,18.0,36.0,50.0,35.5,14.5
22,5,Peter Thiemann,ICFP,2023,8.0,21.0,38.1,21.4,16.7
27,6,Simon J. Gay,ICFP,2023,11.0,22.0,50.0,28.6,21.4
32,7,Ezgi Çiçek,ICFP,2019,7.0,12.0,58.3,16.7,41.7
37,8,Brandon Lucia,OOPSLA,2019,24.0,46.0,52.2,76.9,-24.7
42,9,Jonathan Immanuel Brachthäuser,POPL,2024,16.0,45.0,35.6,0.0,35.6
47,10,Fritz Henglein,POPL,2024,13.0,25.0,52.0,33.3,18.7


## 5. Plot selected source share changes

In [5]:

def format_signed(value, digits=1):
    if pd.isna(value):
        return "n/a"
    return f"{value:+.{digits}f}"


DISPLAY_NAMES = {
    "jonathanbrachthauser": "Jonathan Brachthäuser",
}

SOURCE_ORDER = ["ICFP", "POPL", "OOPSLA", "PLDI"]
SOURCE_SHORT = {"ICFP": "I", "POPL": "P", "OOPSLA": "O", "PLDI": "L"}
SOURCE_COLORS = {
    "ICFP": "#f5bde6",
    "POPL": "#81c8be",
    "OOPSLA": "#8aadf4",
    "PLDI": "#e5c890",
}

APPENDIX_SELECTED_SOURCE_SHARE_FIGURE_OUT = (
    MAIN_TEXT_FIGURES / "selected_pc_source_share_analysis_appendix.pdf"
)
REPORT_APPENDIX_SELECTED_SOURCE_SHARE_FIGURE_OUT = (
    REPORT_FIGURES / "selected_pc_source_share_analysis_appendix.pdf"
)


def display_name(row):
    return DISPLAY_NAMES.get(row["researcher_id"], row["name"])


def source_group(conference):
    if conference in {"OOPSLA", "OOPSLA1", "OOPSLA2"}:
        return "OOPSLA"
    return conference


def selected_case_source_group(conference):
    if conference in {"OOPSLA", "OOPSLA_COMBINED", "OOPSLA1", "OOPSLA2"}:
        return "OOPSLA"
    return conference


def build_selected_source_distribution_rows(panel_source, selected_source):
    records = []
    for _, selected in selected_source.iterrows():
        for event_time in EVENT_TIMES:
            year = int(selected["pc_year"]) + event_time
            researcher_year = panel_source.loc[
                panel_source["researcher_id"].eq(selected["researcher_id"])
                & panel_source["year"].eq(year)
            ].copy()
            if researcher_year.empty:
                counts = pd.Series(dtype=float)
                pc_flags = pd.Series(dtype=float)
                total_citations = np.nan
            else:
                researcher_year["source_group"] = researcher_year["conference"].map(source_group)
                counts = researcher_year.groupby("source_group")["citation_count"].sum()
                pc_flags = researcher_year.groupby("source_group")["pc_status"].max()
                total_citations = counts.sum()

            for source in SOURCE_ORDER:
                citation_count = counts.get(source, 0.0) if pd.notna(total_citations) else np.nan
                source_pc_status = pc_flags.get(source, 0.0) if pd.notna(total_citations) else np.nan
                source_share = (
                    citation_count / total_citations
                    if pd.notna(total_citations) and total_citations > 0
                    else np.nan
                )
                records.append(
                    {
                        "plot_order": selected["plot_order"],
                        "researcher_id": selected["researcher_id"],
                        "name": selected["name"],
                        "conference": selected["conference"],
                        "plot_conference": selected["plot_conference"],
                        "pc_year": selected["pc_year"],
                        "selected_event_label": selected["selected_event_label"],
                        "event_time": event_time,
                        "year": year,
                        "source_group": source,
                        "citation_count": citation_count,
                        "total_citations": total_citations,
                        "source_share": source_share,
                        "source_share_pct": 100 * source_share if pd.notna(source_share) else np.nan,
                        "pc_status_for_source": source_pc_status,
                    }
                )
    return pd.DataFrame(records)


def event_source_counts(distribution_rows, event_time):
    event_rows = distribution_rows.loc[
        distribution_rows["event_time"].eq(event_time)
    ].copy()
    total_values = event_rows["total_citations"].dropna()
    total = total_values.iloc[0] if not total_values.empty else np.nan
    counts = []
    pc_flags = []
    for source_name in SOURCE_ORDER:
        row = event_rows.loc[event_rows["source_group"].eq(source_name)]
        if row.empty:
            counts.append(np.nan)
            pc_flags.append(False)
        else:
            counts.append(row["citation_count"].iloc[0])
            pc_flags.append(bool(row["pc_status_for_source"].fillna(0).iloc[0]))
    return counts, pc_flags, total


def format_event_time_label(event_time):
    if event_time == 0:
        return "$t=0$"
    return f"$t={event_time:+d}$"


def draw_event_minibars(ax, distribution_rows, event_time, selected_source, y_max):
    counts, pc_flags, total = event_source_counts(distribution_rows, event_time)
    x = np.arange(len(SOURCE_ORDER))

    if pd.isna(total):
        ax.text(0.5, 0.55, "n/a", ha="center", va="center", transform=ax.transAxes, fontsize=8)
        ax.set_axis_off()
        return

    clean_counts = [0 if pd.isna(value) else value for value in counts]
    bar_edgecolors = [
        PC_YEAR_RED if source_name == selected_source else "#555555"
        for source_name in SOURCE_ORDER
    ]
    bar_linewidths = [
        1.25 if source_name == selected_source else 0.45
        for source_name in SOURCE_ORDER
    ]
    ax.bar(
        x,
        clean_counts,
        color=[SOURCE_COLORS[source_name] for source_name in SOURCE_ORDER],
        edgecolor=bar_edgecolors,
        linewidth=bar_linewidths,
        width=0.68,
    )
    for x_value, count in zip(x, clean_counts):
        if count > 0:
            ax.text(
                x_value,
                count + y_max * 0.035,
                f"{int(count)}",
                ha="center",
                va="bottom",
                fontsize=6.1,
            )

    labels = [
        SOURCE_SHORT[source_name] + (r" $\star$" if pc_status else "")
        for source_name, pc_status in zip(SOURCE_ORDER, pc_flags)
    ]
    ax.set_ylim(0, y_max)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=6.7)
    ax.set_yticks([])
    ax.tick_params(axis="x", length=0, pad=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_color("#555555")
    ax.spines["bottom"].set_linewidth(0.55)


def draw_selected_source_distribution_timeline(rows, distribution_rows, plot_orders, figure_title):
    from matplotlib.lines import Line2D

    panels = rows.loc[rows["plot_order"].isin(plot_orders), [
        "plot_order",
        "researcher_id",
        "name",
        "conference",
        "plot_conference",
        "pc_year",
        "selected_event_label",
    ]].drop_duplicates().sort_values("plot_order")

    n_panels = len(panels)
    fig_height = max(3.0, 1.16 * n_panels + 1.10)
    fig = plt.figure(figsize=(8.4, fig_height))
    outer_grid = fig.add_gridspec(
        n_panels,
        1,
        left=0.07,
        right=0.985,
        bottom=0.12,
        top=0.84,
        hspace=0.54,
    )

    row_bounds = []
    bottom_event_axes = {}

    for panel_index, (_, panel_row) in enumerate(panels.iterrows()):
        sub_grid = outer_grid[panel_index, 0].subgridspec(
            1,
            6,
            width_ratios=[1.75, 1, 1, 1, 1, 1],
            wspace=0.28,
        )
        title_ax = fig.add_subplot(sub_grid[0, 0])
        title_ax.axis("off")

        case_rows = rows.loc[
            rows["plot_order"].eq(panel_row["plot_order"])
        ].sort_values("event_time")
        case_distribution = distribution_rows.loc[
            distribution_rows["plot_order"].eq(panel_row["plot_order"])
        ].copy()
        selected_source = selected_case_source_group(panel_row["conference"])
        name = display_name(panel_row)
        title_text = f"{name}\n{panel_row['selected_event_label']}"
        title_ax.text(
            0,
            0.5,
            title_text,
            ha="left",
            va="center",
            fontsize=9.2,
            linespacing=1.12,
        )

        all_counts = []
        for event_time in EVENT_TIMES:
            counts, _, _ = event_source_counts(case_distribution, event_time)
            all_counts.extend([value for value in counts if pd.notna(value)])
        max_count = max(all_counts) if all_counts else 1
        y_max = max(1, max_count * 1.38)

        for event_index, event_time in enumerate(EVENT_TIMES, start=1):
            ax = fig.add_subplot(sub_grid[0, event_index])
            draw_event_minibars(ax, case_distribution, event_time, selected_source, y_max)
            if panel_index == n_panels - 1:
                bottom_event_axes[event_time] = ax

        row_axes = [title_ax] + [fig.axes[-i] for i in range(1, len(EVENT_TIMES) + 1)]
        row_y0 = min(axis.get_position().y0 for axis in row_axes)
        row_y1 = max(axis.get_position().y1 for axis in row_axes)
        row_bounds.append((row_y0, row_y1))

    for (upper_y0, _), (_, lower_y1) in zip(row_bounds[:-1], row_bounds[1:]):
        y_value = (upper_y0 + lower_y1) / 2
        fig.add_artist(
            Line2D(
                [0.07, 0.985],
                [y_value, y_value],
                transform=fig.transFigure,
                color=ROW_SEPARATOR,
                lw=0.45,
            )
        )

    for event_time in EVENT_TIMES:
        ax = bottom_event_axes[event_time]
        position = ax.get_position()
        x_center = (position.x0 + position.x1) / 2
        color = SOFT_BLACK
        label = format_event_time_label(event_time)
        if event_time == 0:
            label = "$t=0$ (PC year)"
        fig.text(x_center, 0.055, label, ha="center", va="center", fontsize=8.0, color=color)

    handles = [
        Line2D(
            [0],
            [0],
            marker="s",
            linestyle="",
            markerfacecolor=SOURCE_COLORS[source_name],
            markeredgecolor="none",
            label=source_name,
            markersize=7,
        )
        for source_name in SOURCE_ORDER
    ]
    handles.append(
        Line2D(
            [0],
            [0],
            marker="*",
            linestyle="",
            color=SOFT_BLACK,
            label="PC service in source year",
            markersize=7,
        )
    )

    fig.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.925),
        ncol=5,
        frameon=False,
        columnspacing=1.0,
        handletextpad=0.35,
        fontsize=8.1,
    )
    fig.suptitle(
        figure_title,
        x=0.07,
        y=0.990,
        ha="left",
        fontsize=13,
    )
    return fig


selected_source_distribution_rows = build_selected_source_distribution_rows(
    panel,
    selected_subset,
)

main_plot_orders = [1, 3]
appendix_plot_orders = [2, 4, 5, 6, 7, 8, 9, 10]

fig = draw_selected_source_distribution_timeline(
    selected_source_share_rows,
    selected_source_distribution_rows,
    main_plot_orders,
    "Citation Distributions for Years",
)
fig.savefig(SELECTED_SOURCE_SHARE_FIGURE_OUT, bbox_inches="tight", dpi=450)
fig.savefig(REPORT_SELECTED_SOURCE_SHARE_FIGURE_OUT, bbox_inches="tight", dpi=450)
plt.close(fig)

appendix_fig = draw_selected_source_distribution_timeline(
    selected_source_share_rows,
    selected_source_distribution_rows,
    appendix_plot_orders,
    "Citation Distributions for Years: Additional Cases",
)
appendix_fig.savefig(APPENDIX_SELECTED_SOURCE_SHARE_FIGURE_OUT, bbox_inches="tight", dpi=450)
appendix_fig.savefig(REPORT_APPENDIX_SELECTED_SOURCE_SHARE_FIGURE_OUT, bbox_inches="tight", dpi=450)
plt.close(appendix_fig)

selected_source_share_rows.to_csv(SELECTED_SOURCE_SHARE_ROWS_OUT, index=False)
pc_year_summary.to_csv(SELECTED_SOURCE_SHARE_SUMMARY_OUT, index=False)
selected_source_distribution_rows.to_csv(SELECTED_SOURCE_DISTRIBUTION_ROWS_OUT, index=False)

print(f"wrote {SELECTED_SOURCE_SHARE_FIGURE_OUT.relative_to(PROJECT)}")
print(f"wrote {REPORT_SELECTED_SOURCE_SHARE_FIGURE_OUT.relative_to(PROJECT)}")
print(f"wrote {APPENDIX_SELECTED_SOURCE_SHARE_FIGURE_OUT.relative_to(PROJECT)}")
print(f"wrote {REPORT_APPENDIX_SELECTED_SOURCE_SHARE_FIGURE_OUT.relative_to(PROJECT)}")
print(f"wrote {SELECTED_SOURCE_SHARE_ROWS_OUT.relative_to(PROJECT)}")
print(f"wrote {SELECTED_SOURCE_SHARE_SUMMARY_OUT.relative_to(PROJECT)}")
print(f"wrote {SELECTED_SOURCE_DISTRIBUTION_ROWS_OUT.relative_to(PROJECT)}")


wrote step_4_artifacts/main_text_figures/selected_pc_source_share_analysis.pdf
wrote report_latex/figures/selected_pc_source_share_analysis.pdf
wrote step_4_artifacts/main_text_figures/selected_pc_source_share_analysis_appendix.pdf
wrote report_latex/figures/selected_pc_source_share_analysis_appendix.pdf
wrote step_4_artifacts/summary_tables/selected_pc_source_share_analysis_rows.csv
wrote step_4_artifacts/summary_tables/selected_source_share_summary.csv
wrote step_4_artifacts/summary_tables/selected_pc_source_distribution_rows.csv


## 6. Output preview

In [6]:
display(Markdown(f"`{SELECTED_SOURCE_SHARE_FIGURE_OUT.relative_to(PROJECT)}`"))
display(Markdown(f"`{APPENDIX_SELECTED_SOURCE_SHARE_FIGURE_OUT.relative_to(PROJECT)}`"))
display(Markdown(f"`{SELECTED_SOURCE_SHARE_SUMMARY_OUT.relative_to(PROJECT)}`"))
display(Markdown(f"`{SELECTED_SOURCE_DISTRIBUTION_ROWS_OUT.relative_to(PROJECT)}`"))
display(pc_year_summary)

`step_4_artifacts/main_text_figures/selected_pc_source_share_analysis.pdf`

`step_4_artifacts/main_text_figures/selected_pc_source_share_analysis_appendix.pdf`

`step_4_artifacts/summary_tables/selected_source_share_summary.csv`

`step_4_artifacts/summary_tables/selected_pc_source_distribution_rows.csv`

,plot_order,name,conference,pc_year,selected_conference_citations,all_studied_conference_citations,source_share_pct,baseline_source_share_pct,delta_source_share_pp
2,1,Clément Pit-Claudel,PLDI,2024,13.0,20.0,65.000000,35.294118,29.705882
7,2,Clément Pit-Claudel,OOPSLA_COMBINED,2025,9.0,20.0,45.000000,16.666667,28.333333
12,3,Viktor Kunčak,OOPSLA,2020,20.0,37.0,54.054054,16.666667,37.387387
17,4,Viktor Kunčak,PLDI,2021,18.0,36.0,50.000000,35.483871,14.516129
22,5,Peter Thiemann,ICFP,2023,8.0,21.0,38.095238,21.428571,16.666667
27,6,Simon J. Gay,ICFP,2023,11.0,22.0,50.000000,28.571429,21.428571
32,7,Ezgi Çiçek,ICFP,2019,7.0,12.0,58.333333,16.666667,41.666667
37,8,Brandon Lucia,OOPSLA,2019,24.0,46.0,52.173913,76.923077,-24.749164
42,9,Jonathan Immanuel Brachthäuser,POPL,2024,16.0,45.0,35.555556,0.000000,35.555556
47,10,Fritz Henglein,POPL,2024,13.0,25.0,52.000000,33.333333,18.666667
